# Replay Public Episode Actions Across Seeds in C++

Every completed public Kaggriculture episode contains 719 actions from each player. Running one player's action history under different environment seeds is a quick way to see whether a long plan still works with different towns and farms. Doing this for thousands of episodes in the official Python environment, however, can make evaluation itself the bottleneck.

This Notebook extracts one player's action history from a public episode and replays it across many seeds in a C++ environment. It also runs the same history in the official Python environment and measures both on the same Kaggle CPU session.


## Starting point

I did not write this C++ environment from scratch. It starts from Nikita Lugovoy's Apache-2.0 Notebook, [4000x environment speedup | Kaggriculture](https://www.kaggle.com/code/nikital7/4000x-environment-speedup-kaggriculture?scriptVersionId=342420572).

The version shared here adds three things:

- the default settings from `kaggle-environments==1.32.7`;
- step-by-step state comparisons between the Python and C++ environments;
- a small program for evaluating two agents over many episodes.

The attached source includes `LICENSE`, `NOTICE`, and `source_manifest.json`. These files identify the original public Notebook separately from the changes made here.


## Agent, environment, and episode

In Kaggle's terminology, an agent is a function that receives an `observation` and returns an `action`. The environment applies the actions from both players and creates the next state. Repeating this process until the game ends produces one episode.

```text
observation → agent → action
                         ↓
                  environment
                         ↓
               next observation, reward
```

In the official Python environment, `env.run([agent_0, agent_1])` runs one episode. The C++ version follows the same loop entirely in C++. It does not call Python at every step, which makes large batches of episodes much faster.


## Build and run

The attached source builds with C++17. In a Kaggle Notebook, the source is attached as a Dataset and only the executable is written to `/kaggle/working`.



In [ ]:
%%bash
set -euo pipefail

SOURCE_FILE=$(find /kaggle/input -type f -name evaluate.cpp -print -quit)
test -n "$SOURCE_FILE"
SOURCE=$(dirname "$SOURCE_FILE")
echo "source: $SOURCE"
g++ -O3 -std=c++17 -pthread \
  -I"$SOURCE" \
  "$SOURCE/evaluate.cpp" \
  -o /kaggle/working/evaluate

/kaggle/working/evaluate --list-agents




Two small agents are included:

- `pass`: returns `PASS` at every step;
- `carrot`: buys CARROT seeds, plants, waters, harvests, and sells them.

The next cell uses 5,000 environment seeds. It swaps player positions for every seed, for a total of 10,000 episodes.



In [ ]:
%%bash
/kaggle/working/evaluate \
  --agents carrot pass \
  --episodes 10000 \
  --threads 8 \
  --seed 1




`mean_rewards` contains the average reward of the two agents. `mean_reward_difference` subtracts the second value from the first. `episodes_per_second` depends on the CPU and thread count of the current Notebook session.

These are episode rewards, not the Leaderboard rating or the Notebook Score.


## Replay a public episode's action history

This is the main use case in this Notebook. I use the term action tape for the ordered sequence of 719 actions returned by one player in a public episode.

The next cell downloads one completed public episode and writes the selected player's actions to `episode-102248386-player0.txt`. Change `EPISODE_ID` and `PLAYER` to use another public episode.



In [ ]:
%%bash
set -euo pipefail

SOURCE_FILE=$(find /kaggle/input -type f -name episode_to_tape.py -print -quit)
test -n "$SOURCE_FILE"
SOURCE=$(dirname "$SOURCE_FILE")
EPISODE_ID=102248386
PLAYER=0

python "$SOURCE/episode_to_tape.py" \
  "$EPISODE_ID" \
  --player "$PLAYER" \
  --output "/kaggle/working/episode-${EPISODE_ID}-player${PLAYER}.txt"




The resulting action tape can be supplied where an agent name would normally appear. The recorded actions are then returned in order from step 0. The following comparison runs the tape against `pass` in both player positions.



In [ ]:
import contextlib
import copy
import io
import json
import subprocess
import time
import urllib.request

from IPython.display import Markdown, display

with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
    from kaggle_environments import make


episode_id = 102248386
player = 0
tape_path = f"/kaggle/working/episode-{episode_id}-player{player}.txt"

# Stable C++ measurement: 5,000 seeds with both player positions.
cpp_result = json.loads(subprocess.check_output([
    "/kaggle/working/evaluate",
    "--agents", f"tape:{tape_path}", "pass",
    "--episodes", "10000",
    "--threads", "8",
    "--seed", "1",
], text=True))

# Read the same selected player's public action history for the official runner.
url = f"https://www.kaggleusercontent.com/episodes/{episode_id}.json"
request = urllib.request.Request(url, headers={"User-Agent": "kaggriculture-cpp-notebook/1.0"})
with urllib.request.urlopen(request, timeout=90) as response:
    replay = json.load(response)
if isinstance(replay.get("replay"), str):
    replay = json.loads(replay["replay"])
elif isinstance(replay.get("replay"), dict):
    replay = replay["replay"]
actions = [joint[player].get("action") or {} for joint in replay["steps"][1:]]
assert len(actions) == 719


def read(value, key, default=None):
    if isinstance(value, dict):
        return value.get(key, default)
    getter = getattr(value, "get", None)
    return getter(key, default) if callable(getter) else getattr(value, key, default)


def tape_agent(observation, configuration=None):
    del configuration
    raw_step = read(observation, "step")
    step = int(raw_step) if raw_step is not None else (
        int(read(observation, "day", 0) or 0) * 24
        + int(read(observation, "hour", 0) or 0)
    )
    action = copy.deepcopy(actions[min(max(step, 0), len(actions) - 1)])
    current_player = int(read(observation, "player", 0) or 0)
    farms = list(read(observation, "farms", []) or [])
    expected_hands = len(read(farms[current_player], "hands", []) or [])
    hands = list(action.get("hands") or [])
    hands.extend([["PASS"] for _ in range(max(0, expected_hands - len(hands)))])
    action["hands"] = hands[:expected_hands]
    return action


def official_episode(seed, tape_first):
    environment = make(
        "kaggriculture",
        configuration={"episodeSteps": 720, "seed": seed},
        debug=False,
    )
    environment.run([tape_agent, "pass"] if tape_first else ["pass", tape_agent])


# Warm up imports and environment initialization outside the measurement.
official_episode(0, True)
python_episodes = 8
started = time.perf_counter()
for seed in range(1, python_episodes // 2 + 1):
    official_episode(seed, True)
    official_episode(seed, False)
python_seconds = time.perf_counter() - started
python_eps = python_episodes / python_seconds
cpp_eps = float(cpp_result["episodes_per_second"])
speedup = cpp_eps / python_eps

table = f"""
| environment | episodes | seconds | episode/second | speedup |
|---|---:|---:|---:|---:|
| Official Python | {python_episodes:,} | {python_seconds:.3f} | {python_eps:.3f} | 1.0× |
| C++ | {cpp_result['episodes']:,} | {cpp_result['seconds']:.3f} | {cpp_eps:,.1f} | {speedup:,.0f}× |
"""
display(Markdown(table))
print(f"Official Python: {python_eps:.3f} episode/s | C++: {cpp_eps:.1f} episode/s | {speedup:.0f}x")




The comparison uses:

- the same Kaggle CPU session;
- the same 719-action history from one public episode;
- `pass` as the other agent;
- both player positions;
- no source download, C++ compilation, or Python import time.

The Python measurement uses 8 episodes to keep the cell reasonably short. The C++ measurement uses 10,000 episodes for a stable timing result. `episode/second` is the number of measured episodes divided by elapsed time, and `speedup` is C++ throughput divided by official Python throughput.

This is an end-to-end comparison, not a microbenchmark of one step function. It compares completing episodes through the official `env.run` path with completing episodes through the C++ evaluator.

The public episode JSON is read once during conversion. During evaluation, the numeric action tape is loaded once; both the environment and action lookup then remain in C++. This is especially useful when testing the same action history across many seeds.

The tape reproduces the actions recorded in the public episode. It does not reproduce the original agent. It never reconsiders an action after seeing a new observation. If the farm diverges from the recorded episode, an action may fail because a crop cannot be planted, a harvest is unavailable, or the planned quantity cannot be sold.

That limitation is important, but a fixed action history is still useful for several tasks:

- screen a strong public episode across different seeds;
- compare two action histories under the same conditions;
- change a few actions and measure the result;
- run regression tests after changing the C++ environment.

Results from a fixed action history should remain separate from results produced by an agent that responds to its observation.


## Modify an agent

The central agent function receives the current state, configuration, and player index, then returns one action for the current step. The included `carrot_agent` has this shape:

```cpp
Action carrot_agent(
    const State& state,
    const Config& configuration,
    int player) {
    const Farm& farm = state.farms[player];
    Action action;

    // Read information corresponding to the observation,
    // then construct BUY_SEED, PLANT, WATER, HARVEST, or SELL.

    return action;
}
```

The complete implementation is in `evaluate.cpp`. A simple way to begin is to change only this function, rebuild, and compare the old and new versions with the same seeds, the same other agent, and both player positions.

The C++ `State` also contains internal environment data. An agent must read only information available in the official `observation`. The included `carrot` agent uses only its own farm and public information.



In [ ]:
from pathlib import Path

source = next(
    path for path in Path("/kaggle/input").rglob("evaluate.cpp")
    if (path.parent / "source_manifest.json").is_file()
)
text = source.read_text()
start = text.index("Action carrot_agent(")
end = text.index("\n}\n", start) + 2
print(text[start:end])


## What was compared with Python

Speed is useful only if the environment produces the same result. Actions recorded from the official Python environment were also supplied to the C++ environment, and the state was compared after every step.

- 15 ordinary episodes and 1 boundary episode covering every action;
- 11,520 states in total;
- both farms, market, town, money, unit positions, inventories, and rewards;
- every market price for all 9 products at integer inventory values from `-10,000` through `30,000`.

A finite test corpus is evidence, not a mathematical proof for every possible Python object. The supported scope is the standard two-player, 10×10 environment with default `marketParams`.


## When to use each environment

The official Python environment remains the reference for final checks before submission. The C++ environment is useful for screening many candidates quickly.

```text
screen many candidates in the C++ environment
                       ↓
recheck promising agents in the official Python environment
```

The C++ version does not run an existing Python agent directly. An agent that needs fast evaluation must be implemented in C++. Generic-runner timeouts, logs, and details of Python object behavior are also outside its scope.


## Summary

This Notebook shares a tool for replaying a public episode's action history across many seeds and measuring both rewards and runtime. It is not a method for reconstructing the original agent.

The release clearly credits the original C++ port, updates it for environment 1.32.7, adds step-level comparisons, and provides a small many-episode evaluator. The included agent can be replaced with a rule-based, search-based, or learned C++ agent while keeping the same evaluation interface.


## Sources

- [Kaggriculture: Getting Started](https://github.com/Kaggle/kaggle-environments/blob/master/kaggle_environments/envs/kaggriculture/AGENTS.md)
- [Kaggriculture environment rules](https://github.com/Kaggle/kaggle-environments/blob/master/kaggle_environments/envs/kaggriculture/README.md)
- [Kaggle Environments](https://github.com/Kaggle/kaggle-environments)
- [4000x environment speedup | Kaggriculture](https://www.kaggle.com/code/nikital7/4000x-environment-speedup-kaggriculture?scriptVersionId=342420572)

